In [24]:
import folium
import numpy as np
import pandas as pd
from folium import FeatureGroup, LayerControl
from folium.plugins import HeatMap
from sqlalchemy import create_engine

In [2]:
import os

from dotenv import load_dotenv

# Database credentials are read from environment variables (see .env.example).
load_dotenv()
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.environ["DB_NAME"]
SCHEMA_NAME = os.getenv("DB_SCHEMA", "INMINDAA_project")

engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

TABLE_NAME = "dim_armoede"
# BIJV: TABLE_ARMOEDE = "dim_armoede"

In [32]:
def fetch_armoede_data():
    query = f"""
    SELECT da.a_inw, da.pc4_code, da.longitude, da.latitude, da.g_ink_pi 
    FROM "{SCHEMA_NAME}".{TABLE_NAME} da 
    """
    df = pd.read_sql(query, con=engine)
    return df

In [40]:
def fetch_armoede_pau_data():
    query = f"""
    SELECT da.pc4_code, da.longitude, da.latitude, da.g_pau_hh 
    FROM "{SCHEMA_NAME}".{TABLE_NAME} da 
    """
    df = pd.read_sql(query, con=engine)
    return df

In [226]:
# Laad de datasets
df_inw = fetch_armoede_data()
df_pau = fetch_armoede_pau_data()

# Controleer benodigde kolommen voor df_inw
required_columns_inw = ['pc4_code', 'latitude', 'longitude', 'a_inw']
if not all(col in df_inw.columns for col in required_columns_inw):
    raise ValueError("df_inw mist een of meer vereiste kolommen: pc4_code, latitude, longitude, a_inw")

# Controleer benodigde kolommen voor df_pau
required_columns_pau = ['pc4_code', 'latitude', 'longitude', 'g_pau_hh']
if not all(col in df_pau.columns for col in required_columns_pau):
    raise ValueError("df_pau mist een of meer vereiste kolommen: pc4_code, latitude, longitude, g_pau_hh")

# Verwijder rijen met ontbrekende waarden
df_inw = df_inw.dropna(subset=required_columns_inw)
df_pau = df_pau.dropna(subset=required_columns_pau)

# Combineer datasets op pc4_code
df = df_inw[['pc4_code', 'latitude', 'longitude', 'a_inw']].merge(
    df_pau[['pc4_code', 'g_pau_hh']],
    on='pc4_code',
    how='outer'
)

# Vul ontbrekende waarden met 0 voor heatmap
df['a_inw'] = df['a_inw'].fillna(0)
df['g_pau_hh'] = df['g_pau_hh'].fillna(0)

# Normaliseer de gewichten voor consistente schaling
df['inw_normalized'] = np.log1p(df['a_inw'].clip(lower=0))  # Log-transformatie
df['pau_normalized'] = np.log1p(df['g_pau_hh'].clip(lower=0))

# Maak een kaart van Nederland
nl_map = folium.Map(location=[52.132633, 5.291266], zoom_start=8, tiles='Cartodb Positron')

# Maak lagen
layer1 = FeatureGroup(name='Inwoners Heatmap')
layer2 = FeatureGroup(name='PAU Huishoudens Heatmap')

# Definieer kleurovergangen met fellere kleuren voor hoge frequenties
gradient_inw = {
    0.0: '#fff7ec',  # Zeer lichtoranje
    0.3: '#fdd0a2',  # Lichtoranje
    0.6: '#f16913',  # Warm oranje
    1.0: '#8c2d04'   # Diep rood/oranje
}
gradient_pau = {
    0.0: '#f7fcfd',  # Zeer lichtblauwgroen
    0.3: '#c7e9c0',  # Lichtblauwgroen
    0.6: '#41ab5d',  # Helder groen
    1.0: '#006837'   # Diep teal
}

# Heatmap voor a_inw (rode kleurovergang)
heat_data_inw = [[row['latitude'], row['longitude'], row['inw_normalized']] for idx, row in df.iterrows()]
HeatMap(heat_data_inw, radius=10, blur=12, gradient=gradient_inw, opacity=0.05).add_to(layer1)

# Heatmap voor g_pau_hh (groene kleurovergang)
heat_data_pau = [[row['latitude'], row['longitude'], row['pau_normalized']] for idx, row in df.iterrows()]
HeatMap(heat_data_pau, radius=4, blur=3, gradient=gradient_pau).add_to(layer2)

# # Heatmap voor a_inw (oranje-rode kleurovergang)
# heat_data_inw = [[row['latitude'], row['longitude'], row['inw_normalized']] for idx, row in df.iterrows()]
# HeatMap(heat_data_inw, radius=10, blur=15, gradient=gradient_inw, opacity=0.4).add_to(layer1)

# # Heatmap voor g_pau_hh (blauwgroene kleurovergang)
# heat_data_pau = [[row['latitude'], row['longitude'], row['pau_normalized']] for idx, row in df.iterrows()]
# HeatMap(heat_data_pau, radius=10, blur=15, gradient=gradient_pau, opacity=0.8).add_to(layer2)


# for idx, row in df.iterrows():
#     folium.Marker(
#         location=[row['latitude'], row['longitude']],
#         popup=f"Postcode: {row['pc4_code']}<br>A_INW: {row['inw_normalized']}Postcode: {row['pc4_code']}<br>PAU_zooi: {row['pau_normalized']}",
#         icon=folium.Icon(color='blue', icon='info-sign')
#     ).add_to(nl_map)

# Voeg lagen toe aan de kaart
layer1.add_to(nl_map)
layer2.add_to(nl_map)
LayerControl().add_to(nl_map)

# Sla de kaart op
nl_map.save('nederland_heatmap_feller_2_vars.html')